# LostMa — export only, per language (matière fully resolved)

Writes **three files per language group** under `./data/`, then prunes columns that don't survive the language split and folds in any manual matière corrections. See `docs/PREPROCESSING.md` for the full column-by-column reference.

All matière logic now lives here, so the works file ships a final, analysis-ready `matiere` column. Matière is derived from the **storyverse hierarchy** (the *Matter of …* storyverses), with the local `matter` field as fallback. The works file carries three matière columns:

| column | meaning |
|---|---|
| `matiere` | final single category used downstream (aliased, normalized, fallback applied, manual corrections folded in) |
| `matieres` | raw storyverse-derived set, preserving the DB taxonomy (Antiquity, England, …) |
| `matter_local` | the raw local `Story_matter` field, for comparison |


Dependencies (including `lostma_db` / `heurist-analyser`) are installed once, pinned, when the project conda env is created -- see `environment.yml` and `requirements.txt`, and the Setup section of `README.md`. No install step needed here; an unpinned `!pip install` in the notebook would risk silently pulling a newer `lostma_db` than the one this pipeline was verified against.


In [ ]:
from pathlib import Path
from lostma_db import LostmaDB

def read_credentials(path="credentials"):
    """Read `login` / `pwd` from a local `key = value` file that stays out of git
    (see .gitignore). Copy `credentials.example` to `credentials` and fill in your
    own Heurist login before running this cell -- never hardcode credentials here."""
    creds = {}
    for line in Path(path).read_text().splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            creds[key.strip()] = value.strip()
    return creds["login"], creds["pwd"]

login, pwd = read_credentials()
db = LostmaDB(login, pwd)
db.sync()   # only the first time, or to refresh from Heurist; reads ./lostma.db otherwise


## Data carpentry steps (after download)

Once `db.sync()` has pulled the raw Heurist tables, everything below is carpentry to get from those tables to the three analysis-ready files. See `docs/PREPROCESSING.md` for the full column-by-column reference -- what each output column means, which ones get dropped and why, and the bug this cleanup fixed.

1. **Pull source tables** — fetch `witnesses`, `parts`, `stories` for the target languages, keeping only columns above a 5% fill-rate threshold (plus a fixed allow-list of always-kept columns).
2. **Configure matière resolution** — define the alias map (e.g. Antiquity → Rome) and manual storyverse overrides (e.g. Carolingian → France) for storyverses not yet linked to a Matter in Heurist.
3. **Resolve matière from the storyverse network** — for each work, walk the Story → Storyverse → cycle hierarchy (`matters_for`) to collect every reachable *Matter of X* label, falling back to the local `Story_matter` field when the hierarchy doesn't resolve.
4. **Attach manuscripts and parse dates** — map each witness's parts to their parent document/manuscript, and parse free-text creation-date fields into numeric `start`/`end`/`mid` year columns.
5. **Build the three core tables** — deduplicate into `works` (one row per text, carrying `matiere`/`matieres`/`matter_local`), `manuscripts` (one row per document), and `linkage` (one row per witness, linking work ↔ manuscript).
6. **Verify matière coverage** — tally the final `matiere` distribution and list any works still `Unknown`.
7. **Diagnose unresolved cases** (optional) — trace a specific work's path through the storyverse chain to find where it dead-ends, for manual follow-up.
8. **Split by language group, prune, and write output** — split `works`/`manuscripts`/`linkage` into `french` (Old + Middle French merged) and `german` (Middle High German) groups, drop columns that are essentially empty *within that language group* (the 5%-fill rule from step 1 was computed across all three languages pooled, so a column can pass the pooled threshold yet be 0% filled once split — this step catches those), and write `{lang}_works.xlsx`, `{lang}_manuscripts.xlsx`, `{lang}_linkage.json` to `./data/`.
9. **Fold in manual matière corrections** — if `data/{lang}_works_EdB.xlsx` already exists (a colleague has hand-reviewed the exported works and added an annotation column, e.g. `override` for French, `is_Heldenepik` for German), re-read it, apply the override to `matiere` where applicable, and re-save. Safe to skip on a first run; safe to re-run after fresh manual edits.


## Pull source tables
`stories` carries the Story/Storyverse network used for matière.

In [3]:
available_languages = [
    "fro (Old French)",
    "frm (Middle French)",
    "gmh (Middle High German)",
]

always_keep_these_columns = [
    "TextTable_is_adapted_by H-ID", "TextTable_is_adapted_by Name",
    "TextTable_place_of_creation H-ID", "TextTable_place_of_creation Name",
    "TextTable_is_derived_from H-ID", "TextTable_is_derived_from Name",
    "TextTable_nature_of_derivations",
    "TextTable_is_written_by H-ID", "TextTable_is_written_by Name",
    "TextTable_author_freetext",
    "TextTable_in_stemma H-ID", "TextTable_in_stemma Name",
]

witnesses = db.witnesses(available_languages, always_keep_these_columns)
parts = db.parts(available_languages)
stories = db.stories(available_languages)

Genre parent_genre H-ID
Witness place_of_creation H-ID
TextTable place_of_creation H-ID
Witness scribe H-ID
TextTable is_written_by H-ID
TextTable is_adapted_by H-ID
TextTable in_stemma H-ID
TextTable is_derived_from H-ID
TextTable regional_writing_style H-ID
Witness regional_writing_style H-ID
Story is_part_of_storyverse H-ID
DocumentTable location H-ID
Repository city H-ID
Witness last_observed_in_doc H-ID
Dropping 17 columns with < 5% filled values:
  - Witness_last_observed_in_doc H-ID (0.03%)
  - Witness_last_observed_in_doc Name (0.03%)
  - Witness_claim_freetext (0.00%)
  - Witness_alternative_sigla (0.26%)
  - Witness_regional_writing_style H-ID (0.23%)
  - Witness_regional_writing_style Name (0.23%)
  - Witness_scribe H-ID (0.00%)
  - Witness_scribe Name (0.00%)
  - Witness_number_of_hands (0.00%)
  - Witness_scribe_note (0.00%)
  - Witness_place_of_creation H-ID (3.74%)
  - Witness_place_of_creation Name (3.74%)
  - Witness_place_of_creation_source (0.03%)
  - TextTable_claim

In [4]:
WITNESS_ID = "Witness_H-ID"
PART_COL   = "Witness_observed_on_pages H-ID"
TEXT_ID    = "TextTable_H-ID"
TEXT_NAME  = "TextTable_preferred_name"
STORY_ID   = "Story_H-ID"
LANG       = "TextTable_language_COLUMN"
PART_ID    = "Part_H-ID"
DOC_ID     = "DocumentTable_H-ID"
SHELFMARK  = "DocumentTable_current_shelfmark"

for label, frame, needed in [
    ("witnesses", witnesses, [WITNESS_ID, PART_COL, TEXT_ID, TEXT_NAME, STORY_ID, LANG]),
    ("parts", parts, [PART_ID, DOC_ID, SHELFMARK]),
    ("stories", stories, ["Source_type", "Source_H-ID", "Source_preferred_name",
                          "Storyverse_H-ID", "Storyverse_preferred_name"]),
]:
    missing = [c for c in needed if c not in frame.columns]
    print(label, "OK" if not missing else f"MISSING {missing}")

witnesses OK
parts OK
stories OK


## Matière configuration

`MATIERE_ALIASES` folds source labels onto the levels of interest (Antiquity = Matter of Rome). Matter of England and Matter of Britain are kept **separate** (England = insular English-hero romances; Britain = Arthurian). `STORYVERSE_MATTER_OVERRIDES` bridges storyverses not yet linked to a Matter in Heurist — the Carolingian cycle is the Matter of France but its apex node isn't linked yet. **Remove each override once Paris adds the real `member_of_cycle` link.**

In [5]:
MATIERE_ALIASES = {"antiquity": "Rome"}
LEVELS_OF_INTEREST = {"Britain", "England", "France", "Rome", "Other"}
STORYVERSE_MATTER_OVERRIDES = {"carolingian": "France"}

def apply_alias(label):
    if label is None:
        return None
    return MATIERE_ALIASES.get(str(label).strip().lower(), str(label).strip())

def normalize_matiere(label):
    label = apply_alias(label)
    return label if label in LEVELS_OF_INTEREST else "Other"

def final_matiere(sv_list, local_list):
    if sv_list:
        return apply_alias(sorted(sv_list)[0])
    if local_list:
        return normalize_matiere(sorted(local_list)[0])
    return "Unknown"

## Resolve matière from the storyverse network
`matters_for(story_id)` walks up Story -> Storyverse -> cycle -> ... and returns every *Matter of X* (or override) it reaches; sets keep hybrids.

In [6]:
import re
import pandas as pd
import numpy as np

MATTER_RE = re.compile(r"^\s*(?:matter of|mati[eè]re de)\s+(.+)$", re.I)

def to_id(x):
    try:
        return int(float(x))
    except (TypeError, ValueError):
        return None

def build_matter_resolver(stories_df, overrides=None):
    overrides = {k.lower(): v for k, v in (overrides or {}).items()}
    parents, name = {}, {}
    for _, r in stories_df.iterrows():
        s, sv = r["Source_H-ID"], r["Storyverse_H-ID"]
        name[s] = r["Source_preferred_name"]
        if pd.notna(sv):
            name[sv] = r["Storyverse_preferred_name"]
            parents.setdefault(s, set()).add(sv)
    matter_label = {}
    for h, n in name.items():
        m = MATTER_RE.match(str(n))
        if m:
            matter_label[h] = m.group(1).strip()
        elif str(n).strip().lower() in overrides:
            matter_label[h] = overrides[str(n).strip().lower()]
    def matters_for(node, max_depth=20):
        seen, stack, found = set(), [(node, 0)], set()
        while stack:
            nid, d = stack.pop()
            if nid in seen or d > max_depth:
                continue
            seen.add(nid)
            if nid in matter_label:
                found.add(matter_label[nid])
            for p in parents.get(nid, ()):
                stack.append((p, d + 1))
        return sorted(found)
    return matters_for

matters_for = build_matter_resolver(stories, STORYVERSE_MATTER_OVERRIDES)

ts = witnesses[[TEXT_ID, STORY_ID]].copy()
ts["story_id"] = ts[STORY_ID].map(to_id)
ts = ts.dropna(subset=[TEXT_ID, "story_id"]).drop_duplicates()
story_matter = {sid: matters_for(sid) for sid in ts["story_id"].unique()}
ts["matter"] = ts["story_id"].map(story_matter)
matter_by_text = ts.groupby(TEXT_ID)["matter"].apply(lambda L: sorted({m for x in L for m in x}))

## Attach manuscript and parse dates

In [7]:
part_to_doc = (parts[[PART_ID, DOC_ID, SHELFMARK]]
               .dropna(subset=[PART_ID]).drop_duplicates(PART_ID).set_index(PART_ID))

def as_list(x):
    if isinstance(x, (list, np.ndarray)):
        return list(x)
    if pd.isna(x):
        return []
    return [x]

def parse_year_range(v):
    if pd.isna(v):
        return (pd.NA, pd.NA)
    nums = re.findall(r'-?\d+', str(v))
    if not nums:
        return (pd.NA, pd.NA)
    return (int(nums[0]), int(nums[0]) if len(nums) == 1 else int(nums[1]))

wp = witnesses.assign(_part=witnesses[PART_COL].apply(as_list)).explode('_part')
wp[DOC_ID] = wp['_part'].map(part_to_doc[DOC_ID])
wp[SHELFMARK] = wp['_part'].map(part_to_doc[SHELFMARK])

for base in ['Witness_date_of_creation', 'TextTable_date_of_creation']:
    if base in wp.columns:
        parsed = wp[base].apply(parse_year_range)
        wp[f'{base}_start'] = pd.array([p[0] for p in parsed], dtype='Int64')
        wp[f'{base}_end'] = pd.array([p[1] for p in parsed], dtype='Int64')
        wp[f'{base}_mid'] = ((wp[f'{base}_start'] + wp[f'{base}_end']) / 2).round().astype('Int64')

print("witness-part rows:", len(wp), "| unlinked to document:", int(wp[DOC_ID].isna().sum()))

witness-part rows: 3734 | unlinked to document: 1


## Build works, manuscripts, linkage

In [8]:
story_cols = [c for c in witnesses.columns if c.startswith('Story_')]
text_cols = [c for c in witnesses.columns if c.startswith('TextTable_')]
text_dates = [f'TextTable_date_of_creation_{s}' for s in ['start', 'end', 'mid'] if f'TextTable_date_of_creation_{s}' in wp.columns]

works = wp.drop_duplicates(subset=[TEXT_ID])[text_cols + text_dates].reset_index(drop=True)

works['matieres'] = works[TEXT_ID].map(matter_by_text).apply(lambda v: v if isinstance(v, list) else [])

matter_local = (witnesses[[TEXT_ID, 'Story_matter']].dropna()
                .groupby(TEXT_ID)['Story_matter']
                .apply(lambda s: sorted(set(map(str, s)))))
works['matter_local'] = works[TEXT_ID].map(matter_local).apply(lambda v: v if isinstance(v, list) else [])

works['matiere'] = [final_matiere(sv, loc) for sv, loc in zip(works['matieres'], works['matter_local'])]

def flat_unique(series):
    out = []
    for v in series:
        if isinstance(v, (list, np.ndarray)):
            out += [x for x in v if pd.notna(x)]
        elif pd.notna(v):
            out.append(v)
    return list(dict.fromkeys(map(str, out)))

story_agg = (witnesses[[TEXT_ID] + story_cols].groupby(TEXT_ID).agg(
    storyverses=('Story_is_part_of_storyverse Name', flat_unique),
    stories=('Story_preferred_name', flat_unique),
).reset_index())
works = works.merge(story_agg, on=TEXT_ID, how='left')

manuscripts = (parts.dropna(subset=[DOC_ID]).drop_duplicates(DOC_ID)
               .drop(columns=[PART_ID, 'Part_div_order', 'Part_page_ranges'], errors='ignore')
               .reset_index(drop=True))

link_cols = {
    WITNESS_ID: 'witness_id', TEXT_ID: 'work_id', TEXT_NAME: 'work',
    DOC_ID: 'manuscript_id', SHELFMARK: 'shelfmark',
    'Witness_preferred_siglum': 'siglum', 'Witness_status_witness': 'status',
    'Witness_is_excerpt': 'is_excerpt',
    'Witness_date_of_creation_start': 'date_start',
    'Witness_date_of_creation_end': 'date_end',
    'Witness_date_of_creation_mid': 'date_mid',
    LANG: 'language',
}
link_cols = {k: v for k, v in link_cols.items() if k in wp.columns}
linkage = (wp.dropna(subset=[DOC_ID]).drop_duplicates(subset=[WITNESS_ID, DOC_ID])
           [list(link_cols)].rename(columns=link_cols))

print("works:", works.shape, "| manuscripts:", manuscripts.shape, "| linkage:", linkage.shape)

works: (456, 38) | manuscripts: (2494, 15) | linkage: (3473, 12)


## Verify matière coverage

In [9]:
print("matière distribution (final column):")
print(works['matiere'].value_counts(dropna=False).to_string())

unknown = works[works['matiere'] == 'Unknown']
print(f"\nstill Unknown: {len(unknown)} works")
for _, r in unknown[[TEXT_NAME]].head(20).iterrows():
    print("  ", r[TEXT_NAME])

matière distribution (final column):
matiere
France     162
Other      102
Britain     91
Rome        77
Unknown     13
England     11

still Unknown: 13 works
   Athis et Prophilias
   Alexandre, 1re et 2e rédactions en prose
   Alexandre, 3e rédaction en prose
   Beaudous
   Girart de Roussillon, abrégé
   Guillaume de Palerne
   Le roman de Balain
   Apollonius de Tyr
   Apollonius de Tyr
   Apollonius de Tyr
   Apollonius de Tyr
   Aventures des bruns
   Apollonius de Tyr


## Diagnostic: trace a text's matière path
Run on any title still showing Unknown to see where the storyverse chain breaks (dead-end cycle, unlinked story, or no story) — that tells Paris the exact fix.

In [10]:
def trace_text_matiere(query, witnesses, stories, overrides=None):
    overrides = {k.lower(): v for k, v in (overrides or {}).items()}
    if str(query).isdigit():
        sel = witnesses[witnesses[TEXT_ID].map(to_id) == int(query)]
    else:
        sel = witnesses[witnesses[TEXT_NAME].astype(str).str.contains(str(query), case=False, na=False)]
    if sel.empty:
        print(f"no text matching {query!r}"); return
    tid, tname = sel[TEXT_ID].iloc[0], sel[TEXT_NAME].iloc[0]
    print(f"TEXT: {tname} (H-ID {tid})")
    parents, name = {}, {}
    for _, r in stories.iterrows():
        s, sv = r["Source_H-ID"], r["Storyverse_H-ID"]
        name[s] = r["Source_preferred_name"]
        if pd.notna(sv):
            name[sv] = r["Storyverse_preferred_name"]; parents.setdefault(s, set()).add(sv)
    matter_label = {}
    for h, n in name.items():
        m = MATTER_RE.match(str(n))
        if m:
            matter_label[h] = m.group(1).strip()
        elif str(n).strip().lower() in overrides:
            matter_label[h] = overrides[str(n).strip().lower()] + " [override]"
    st = sel[[STORY_ID, "Story_preferred_name"]].dropna(subset=[STORY_ID]).drop_duplicates()
    if st.empty:
        print("  no story linked -> matière underivable"); return
    for _, sr in st.iterrows():
        sid, sname = to_id(sr[STORY_ID]), sr["Story_preferred_name"]
        print(f"  STORY: {sname} (H-ID {sid})")
        if sid not in parents and sid not in matter_label:
            print("     not part of any storyverse  ->  DEAD END (link story to a cycle)"); continue
        reached, deadends, seen, stack = [], [], set(), [(sid, [name.get(sid, sid)])]
        while stack:
            nid, path = stack.pop()
            if nid in seen: continue
            seen.add(nid)
            if nid in matter_label:
                reached.append(path + [f"<{matter_label[nid]}>"]); continue
            ps = parents.get(nid)
            if not ps:
                if nid != sid: deadends.append(path)
                continue
            for p in ps: stack.append((p, path + [name.get(p, p)]))
        if reached:
            for p in reached: print("     OK reaches:", " -> ".join(map(str, p)))
        else:
            print("     NO matter reachable")
            for p in (deadends or [[name.get(sid, sid)]]):
                print("       dead-ends at:", " -> ".join(map(str, p)))

print("Matter storyverses present:",
      sorted({n for n in stories["Storyverse_preferred_name"].dropna().unique() if MATTER_RE.match(str(n))}))
for q in ["Roland", "Aspremont", "Saisnes"]:
    trace_text_matiere(q, witnesses, stories, STORYVERSE_MATTER_OVERRIDES)

Matter storyverses present: ['Matter of Antiquity', 'Matter of Britain', 'Matter of England', 'Matter of France']
TEXT: Pfaffe Konrad: 'Rolandslied' (H-ID 56255)
  STORY: Roland (H-ID 941)
     OK reaches: Roland -> Roncevaux -> Charlemagne -> Carolingian -> <France [override]>
TEXT: Aspremont (H-ID 596)
  STORY: Aspremont (H-ID 95)
     OK reaches: Aspremont -> Agolant -> Charlemagne -> Carolingian -> <France [override]>
TEXT: Saisnes (H-ID 752)
  STORY: Saxons (H-ID 166)
     OK reaches: Saxons -> Charlemagne – suite Roncevaux -> Roncevaux -> Charlemagne -> Carolingian -> <France [override]>


## Writers

In [11]:
from openpyxl.styles import Font
from openpyxl.utils import get_column_letter

def to_cell(v):
    if isinstance(v, (list, np.ndarray)):
        return '; '.join(str(x) for x in v)
    return v

def col_width(series, header):
    m = series.astype(str).str.len().max()
    longest = max(int(m) if pd.notna(m) else 0, len(str(header)))
    return min(60, max(12, longest + 2))

def write_xlsx(df, path, sheet):
    out = df.apply(lambda s: s.map(to_cell))
    with pd.ExcelWriter(path, engine='openpyxl') as xl:
        out.to_excel(xl, index=False, sheet_name=sheet)
        ws = xl.sheets[sheet]
        ws.freeze_panes = 'A2'
        for j, c in enumerate(out.columns, 1):
            ws.cell(row=1, column=j).font = Font(name='Arial', bold=True)
            ws.column_dimensions[get_column_letter(j)].width = col_width(out[c], c)

## Prune columns that don't survive the language split

`always_keep_these_columns` in step 1 was needed so a column relevant to *either* language survives the initial pull, but that means columns can ride along into a language group where they add nothing -- e.g. German inherited 16 completely empty `TextTable_*` columns this way (verse/rhyme/stanza type, author/adaptor fields, regional writing style, ...). `prune_sparse_columns` re-checks fill rate *within* each language group, drops anything below 5% that isn't one of the columns every downstream step depends on, and prints exactly what it dropped and why. See `docs/PREPROCESSING.md` for the before/after column counts.


In [ ]:
CORE_WORK_COLUMNS = {
    TEXT_ID, TEXT_NAME, LANG, "TextTable_literary_form",
    "TextTable_date_of_creation", "TextTable_date_of_creation_start",
    "TextTable_date_of_creation_end", "TextTable_date_of_creation_mid",
    "matiere", "matieres", "matter_local", "storyverses", "stories",
    "override", "is_Heldenepik",  # manual-annotation columns (step 9) -- protected here
    # too, in case this step is ever reordered to run after they exist.
    # Currently a no-op: at this point in the pipeline neither column exists
    # yet (they are added by hand in Excel, after this step runs), and
    # is_Heldenepik/override are both well above the 5% fill threshold anyway.
}
CORE_MANUSCRIPT_COLUMNS = {DOC_ID, SHELFMARK}
MIN_FILL = 0.05  # same threshold as the pooled pull in step 1, applied per language group

def prune_sparse_columns(df, keep, min_fill=MIN_FILL, label=""):
    fill = df.notna().mean()
    drop = [c for c in df.columns if c not in keep and fill.get(c, 0) < min_fill]
    if drop:
        print(f"{label}: dropping {len(drop)} column(s) <{min_fill:.0%} filled "
              f"within this language group: {drop}")
    return df.drop(columns=drop)


## Write three files per language group

In [ ]:
import json
from pathlib import Path

OUT = Path("data")
OUT.mkdir(parents=True, exist_ok=True)

language_groups = {
    "french": ["fro (Old French)", "frm (Middle French)"],
    "german": ["gmh (Middle High German)"],
}

for name, langs in language_groups.items():
    works_l = works[works[LANG].isin(langs)].reset_index(drop=True)
    link_l = linkage[linkage['language'].isin(langs)].reset_index(drop=True)
    doc_ids = set(link_l['manuscript_id'].dropna())
    ms_l = manuscripts[manuscripts[DOC_ID].isin(doc_ids)].reset_index(drop=True)

    works_l = prune_sparse_columns(works_l, CORE_WORK_COLUMNS, label=f'{name} works')
    ms_l = prune_sparse_columns(ms_l, CORE_MANUSCRIPT_COLUMNS, label=f'{name} manuscripts')

    write_xlsx(works_l, OUT / f'{name}_works.xlsx', 'works')
    write_xlsx(ms_l, OUT / f'{name}_manuscripts.xlsx', 'manuscripts')
    link_l.to_json(OUT / f'{name}_linkage.json', orient='records', indent=2, force_ascii=False)

    print(f"{name}: {len(works_l)} works, {len(ms_l)} manuscripts, {len(link_l)} appearances")

## Fold in manual matière corrections

A colleague reviews `{lang}_works.xlsx` by hand in Excel and saves an annotated copy as `{lang}_works_EdB.xlsx` -- currently an `override` column for French (a corrected `matiere` value, e.g. Cliges: Rome to Britain) and `is_Heldenepik` for German (a genre flag, not a matiere override -- see `docs/PREPROCESSING.md` for why the two are handled differently). `MANUAL_MATIERE_COLUMNS` says, per language, which column (if any) should override `matiere`; only rows where that column is filled in are touched, and the manual column itself is always preserved.

**This step used to live in the analysis notebook** (`01-networks.ipynb`, `apply_edb_override`), reading a column called `EdB` that never actually existed -- French's manual corrections were silently never applied. Doing the merge here, once, keeps `matiere` unambiguous everywhere downstream.


In [ ]:
MANUAL_MATIERE_COLUMNS = {"french": "override", "german": None}  # column that overrides `matiere`, per language; None = no matiere override for this language

for name in language_groups:
    path = OUT / f"{name}_works_EdB.xlsx"
    override_col = MANUAL_MATIERE_COLUMNS.get(name)
    if not path.exists():
        print(f"{name}: no {path.name} yet -- nothing to fold in (colleague has not reviewed it)")
        continue

    w = pd.read_excel(path)
    if override_col and override_col in w.columns:
        has_override = w[override_col].notna() & (w[override_col].astype(str).str.strip() != "")
        corrected = w.loc[has_override, override_col].map(normalize_matiere)
        changed = (w.loc[has_override, "matiere"] != corrected).sum()
        w.loc[has_override, "matiere"] = corrected
        label = "manual " + repr(override_col) + " correction(s)"
        print(f"{name}: applied {has_override.sum()} {label} from {path.name} "
              f"({changed} actually changed the resolved matiere)")
    else:
        print(f"{name}: {path.name} has no matiere-override column to fold in "
              f"(present but untouched: {list(w.columns)})")

    write_xlsx(w, path, "works")
